In [ ]:
from PIL import Image
import torch
import torchvision.models as models
import torchvision.transforms as transforms
import numpy as np
import os
import pickle
from sklearn.decomposition import PCA
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm

In [ ]:
# Update this path to point to your local data directory
DATA_PATH = "./data"

In [3]:
# Verifica se MPS è disponibile
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

In [ ]:
model = models.resnet50(pretrained=True)
model.to(device)
model.eval()

In [5]:
preprocess = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

In [6]:
class PatchDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.image_paths = []
        self.labels = []
        for label_dir in os.listdir(root_dir):
            if os.path.isdir(os.path.join(root_dir, label_dir)):
                for image_name in os.listdir(os.path.join(root_dir, label_dir)):
                    if image_name.startswith('._'):
                        continue  # Ignora i file nascosti
                    self.image_paths.append(os.path.join(root_dir, label_dir, image_name))
                    self.labels.append(label_dir)

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image_path = self.image_paths[idx]
        label = self.labels[idx]
        image = Image.open(image_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, label

In [7]:
def save_embeddings(model, fname, dataloader, dataset=None, is_imagefolder=False, 
                    save_patches=False, sprite_dim=128, overwrite=False):
    
    if os.path.isfile('%s.pkl' % fname) and (overwrite == False):
        return None

    embeddings, labels = [], []
    patches = []

    for batch, target in tqdm(dataloader):
        if save_patches:
            for img in batch:
                patches.append(transforms.ToPILImage()(img).resize((sprite_dim, sprite_dim)))
        
        with torch.no_grad():
            batch = batch.to(device)
            embeddings.append(model(batch).detach().cpu().numpy())
            labels.append(np.array(target))

    embeddings = np.vstack(embeddings)
    labels = np.hstack(labels).squeeze()

    if is_imagefolder:
        id2label = dict(map(reversed, dataset.class_to_idx.items()))
        labels = np.array(list(map(id2label.get, labels.ravel())))
    
    asset_dict = {'embeddings': embeddings, 'labels': labels}
    if save_patches:
        asset_dict.update({'patches': patches})
    with open('%s.pkl' % (fname), 'wb') as handle:
        pickle.dump(asset_dict, handle, protocol=pickle.HIGHEST_PROTOCOL)

In [8]:
dataset = PatchDataset(root_dir=f'{DATA_PATH}/batches/', transform=preprocess)
dataloader = DataLoader(dataset, batch_size=32, shuffle=False)

In [ ]:
save_embeddings(model, 'resnet50', dataloader, is_imagefolder=False, save_patches=False, sprite_dim=128, overwrite=True)